## PPO Understanding

### Before PPO?
- While using classic Policy Gradient Algorithm training was unstable as it's objective was *"If an action produced high-reward, increase its probability"* as model follwed high-reward it can abruptly change its policy which lead to collapse in training, massive variance.

#### TRPO(Trust Region Policy Optimization)
- It introduced *"Don't allow the policy to move too far in one update"* meaning a threshold was set that policy can change upto this only, but implementing this was very complex.

### PPO(Proximal Policy Optimization)
- It introduced *"Let Gradient Descent improve the policy, but automatically ignore updates that try to change the policy too much."*
- PPO is an **on-policy algorithm** because it uses data collected by the current(or very recent) policy, and it prevents that policy from drifting too far while reusing the same batch.

## PPO Implementation

In [1]:
import torch
import torch.nn as nn 
import torch.optim as optim 
import torch.nn.functional as F
import numpy as np 
import gymnasium as gym 
import wandb
import imageio
from collections import deque

In [11]:
class Config:
    def __init__(self):
        self.env_name = "CartPole-v1"
        self.total_timesteps = 100000
        self.learning_rate = 7.0e-4
        self.gamma = 0.99
        self.gae_lambda = 0.95
        self.clip_epsilon = 0.2
        self.epochs = 4
        self.batch_size = 128
        self.buffer_size = 2048
        self.hidden_size = 64
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.entropy_coef = 0.01
        self.video_log_interval = 20
        
config = Config()
wandb.init(project = "ppo-cartpole", config = vars(config), mode = 'online')
        

In [12]:
class WandbVideoRecorder(gym.Wrapper):
    """ 
        This wrapper will grab the RBG frames at every step and at the end of an episode
        will stitch those frames into an .mp4
    """
    def __init__(self,env, interval = 20):
        super().__init__(env)   
        self.interval = interval 
        self.episode_count = 0
        self.recording = False 
        self.frames = []

    def _get_render_frame(self):
        """ 
            Helper to safely extract the RGB array from render()
        """
        frame = self.env.render()

        if isinstance(frame, dict):
            frame = frame.get('rgb', None)

        #Ensure it's a proper numpy array
        if frame is not None:
            frame = np.array(frame, dtype = np.uint8)
        return frame
    
    def reset(self, **kwargs):
        if self.episode_count % self.interval == 0:
            self.recording = True 
            self.frames = []
        else:
            self.recording = False 
            
        obs, info = self.env.reset(**kwargs)
        
        #If recording, grab the first frame
        if self.recording:
            frame = self._get_render_frame()
            if frame is not None:
                self.frames.append(frame)
                
        return obs, info 
    
    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        
        if self.recording:
            frame = self._get_render_frame()
            if frame is not None:
                self.frames.append(frame)

        done = terminated or truncated
        if done:
            if self.recording:
                self.save_and_log_video()
            else:
                self.episode_count += 1
                
        return obs, reward, terminated, truncated, info 
    
    def save_and_log_video(self):
        if self.recording and len(self.frames) > 0:
            video_path = f"cartpole_ep_{self.episode_count}.mp4"
            imageio.mimsave(video_path, self.frames, fps = 30, macro_block_size = 1)
        
            wandb.log({
                "gameplay_video": wandb.Video(video_path, fps = 30, format = "mp4"),
                "episode": self.episode_count
            })
            print(f"-------Successfully logged videos for Episode {self.episode_count}----")
        self.episode_count += 1
        self.recording = False 
        self.frames = []
        

In [13]:
class ActorCritic(nn.Module):
    def __init__(self, obs_dim, num_actions):
        super(ActorCritic, self).__init__()
        
        #------------CNN Based Feature Extractor-------------
        #Input: (4, 84, 84) -> Output: (64, 7, 7) --> Flatten --> 3136
        self.shared = nn.Sequential(
            nn.Linear(obs_dim, config.hidden_size),
            nn.Tanh(),
            nn.Linear(config.hidden_size, config.hidden_size),
            nn.Tanh()
        )
        
        self.actor = nn.Linear(config.hidden_size, num_actions)
        # self.actor_logstd = nn.Parameter(torch.zeros(action_dim))
        
        self.critic = nn.Linear(config.hidden_size, 1)
        
    
    def forward(self, x):
        features = self.shared(x) 
        #Trick Divide Logits by sqrt(num_actions) for stable initialization
        logits = self.actor(features) / torch.sqrt(torch.tensor(2.0, device = config.device))
        value = self.critic(features)
        return logits, value
    
    def get_action(self, obs):
        #Obs comes in as an Numpy Array
        obs_tensor = torch.FloatTensor(obs).unsqueeze(0).to(config.device)  #We also add batch_dim SHAPE:[1, 4, 84, 84]
        logits, value = self.forward(obs_tensor)
        #Use categorical distribution instead of Normal
        dist = torch.distributions.Categorical(logits = logits)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        return action.item(), log_prob.item(), value.item()
    
    def evaluate(self, obs, action):
        logits, value = self.forward(obs)
        dist = torch.distributions.Categorical(logits = logits)
        log_prob = dist.log_prob(action)
        entropy = dist.entropy()
        return log_prob, value.squeeze(-1), entropy    
        

In [14]:
class RolloutBuffer:
    def __init__(self):
        self.obs = []
        self.actions = []
        self.rewards = []
        self.dones = []
        self.log_probs = []
        self.values = []
        
    def add(self, obs, action, reward, done, log_prob, value):
        self.obs.append(obs)
        self.actions.append(action)
        self.rewards.append(reward)
        self.dones.append(done)
        self.log_probs.append(log_prob)
        self.values.append(value)
        
    def get(self):
        data = {
            "obs": torch.FloatTensor(np.array(self.obs)).to(config.device),
            "actions": torch.LongTensor(np.array(self.actions)).to(config.device),
            "rewards": torch.FloatTensor(np.array(self.rewards)).to(config.device),
            "dones": torch.FloatTensor(np.array(self.dones)).to(config.device),
            "log_probs": torch.FloatTensor(np.array(self.log_probs)).to(config.device),
            "values": torch.FloatTensor(np.array(self.values)).to(config.device)
        }
        
        self.clear()
        return data 
    
    def clear(self):
        self.obs, self.actions, self.rewards = [], [], []
        self.dones, self.log_probs, self.values = [], [], []

#### GAE(Generalized Advantage Estimation)
- Temporal Difference Error represents the immediate surprise in reward plus discounted future value.
- GAE balances variance and bias by taking an exponentially weighted average of k-step advantages.

In [15]:
def compute_gae(buffer_data, last_value):
    #Get the trajectory data collected during the rollout
    rewards = buffer_data["rewards"]    #This is the immediate rewards r_t
    values = buffer_data["values"]      #This is the Critic's esitmated state values V(s_t)
    dones = buffer_data["dones"]        #This is the termination flags
    
    #Initialize the advantage tensor with zeros matching the shape of the Rewards tensor
    advantages = torch.zeros_like(rewards).to(config.device)
    last_gae = 0
    
    #Iterate backwards through time: t = T-1, T-2, T-3,.....,0
    for t in reversed(range(len(rewards))):
 
        #Determine the Value{s_{t+1}} for the next step
        if t == len(rewards) - 1:
            next_value = last_value     #Value of the state reached after the final step
        else:
            next_value = values[t + 1]
        
        #Calculate Temporal Difference(TD) error: delta_at_t = reward_at_t + (gamma * Value_at_step_t+1 *(1 - done_at_t)) - Value_at_step_t
        delta = rewards[t] + config.gamma * next_value * (1 - dones[t]) - values[t]
        last_gae = delta + config.gamma * config.gae_lambda * (1 - dones[t]) * last_gae
        advantages[t] = last_gae
        
    returns = advantages + values 
    return advantages , returns  

def ppo_update(policy, optimizer, buffer_data, advantages, returns):
    obs = buffer_data["obs"]
    actions = buffer_data["actions"]
    old_log_probs = buffer_data["log_probs"]
    
    dataset_size = len(obs)
    indices = np.arange(dataset_size)
    
    policy_losses, value_losses, entropies = [], [], []
    
    #Mini-Batch Optimization Loop
    for _ in range(config.epochs):
        np.random.shuffle(indices)
        
        #Slice the dataset into mini-batches
        for start in range(0, dataset_size, config.batch_size):
            end = start + config.batch_size
            batch_idx = indices[start:end]
            
            #Slice mini-batch data 
            b_obs = obs[batch_idx]
            b_actions = actions[batch_idx]
            b_old_log_probs = old_log_probs[batch_idx]
            b_advantages = advantages[batch_idx]
            b_returns = returns[batch_idx]

            #Normalize Advantages to have mean 0 and standard deviation 1 to keep gradient updates consistent
            b_advantages = (b_advantages - b_advantages.mean()) / (b_advantages.std() + 1e-8)
    
            
            log_prob, value, entropy = policy.evaluate(b_obs, b_actions)
            
            #Compute probability-ratio r_t(theta)
            ratio = torch.exp(log_prob - b_old_log_probs)
            
            #Unclipped objective element
            surr1 = ratio * b_advantages 
            
            #Clipped objective element
            surr2 = torch.clamp(ratio, 1 - config.clip_epsilon, 1 + config.clip_epsilon) * b_advantages
            
            #PPO Clipped Surrogate Loss
            policy_loss = -torch.min(surr1, surr2).mean()
            
            #Value function i.e clipped for stable training
            value_loss = 0.5 * nn.MSELoss()(value, b_returns)
            
            #Mean Policy Entropy
            entropy_loss = entropy.mean()
            
            #Combined Loss Function in which model will try to minimize the policy,value loss while maximizing the entropy
            loss = policy_loss + 0.5 * value_loss - config.entropy_coef * entropy_loss
            
            optimizer.zero_grad()
            loss.backward()
            
            #Gradient clipping
            nn.utils.clip_grad_norm_(policy.parameters(), 0.5)
            optimizer.step()
            
            policy_losses.append(policy_loss.item())
            value_losses.append(value_loss.item())
            entropies.append(entropy_loss.item())
            
            
    return np.mean(policy_losses), np.mean(value_losses), np.mean(entropies)        

In [16]:
def train():
    
    env = gym.make(config.env_name, render_mode = "rgb_array")
    # env = FrameSkipWrapper(env, skip = config.frame_skip)
    # env = RewardShaper(env)
    env = WandbVideoRecorder(env, interval = config.video_log_interval)
    # env = ImagePreprocessingWrapper(env, frame_stack = config.frame_stack)

    obs_dim = env.observation_space.shape[0]
    num_actions = env.action_space.n
    
    policy = ActorCritic(obs_dim, num_actions).to(config.device)
    optimizer = optim.Adam([
        {'params': policy.shared.parameters(), 'lr': config.learning_rate},
        {'params': policy.actor.parameters(), 'lr': config.learning_rate},
        {'params': policy.critic.parameters(), 'lr': config.learning_rate * 3}
    ], lr = config.learning_rate)
    buffer = RolloutBuffer()
    
    obs, _ = env.reset()
    episode_reward = 0
    episode_count = 0

    reward_window = deque(maxlen = 100)
    
    print(f"Starting CartPole training for {config.total_timesteps} timesteps...")
    print(f"Using Device: {config.device}")
    
    for timestep in range(1, config.total_timesteps + 1):

        # #Linear Learning-Rate Decay
        # frac = 1.0 - (timestep - 1.0) / config.total_timesteps
        # lrnow = frac * config.learning_rate
        # optimizer.param_groups[0]["lr"] = lrnow
        
        action, log_prob, value = policy.get_action(obs)
        
        next_obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated 
        
        buffer.add(obs, action, reward, done, log_prob, value)
        
        obs = next_obs 
        episode_reward += reward 
        
        #Update PPO
        if timestep % config.buffer_size == 0:
            
            # current_entropy_coef = 0.01 - (0.01 - 0.0005) * (timestep / config.total_timesteps)

            with torch.no_grad():
                obs_tensor = torch.FloatTensor(obs).unsqueeze(0).to(config.device)
                _, last_value = policy.forward(obs_tensor)
                last_value = last_value.cpu().item()
                
            buffer_data = buffer.get()
            advantages, returns = compute_gae(buffer_data, last_value)
            
            avg_pol_loss, avg_val_loss, avg_entropy = ppo_update(policy, optimizer, buffer_data, advantages, returns)
            
            wandb.log({
                "timesteps": timestep,
                "update/policy_loss": avg_pol_loss,
                "update_value_loss": avg_val_loss,
                "update_entropy": avg_entropy
            })

        if timestep % 10000 == 0:
            ckpt_path = f"ppo_cartpole_step_{timestep}.pth"
            torch.save(
                {
                    "timestep": timestep,
                    "model_state_dict": policy.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                },
                ckpt_path
            )
            print(f"Saved checkpoint at step: {timestep} -> {ckpt_path}")
        if done:
            episode_count += 1

            reward_window.append(episode_reward)

            rolling_avg = np.mean(reward_window)
            wandb.log({
                "episode": episode_count,
                "episode_reward" : episode_reward,
                "reward_100_avg": rolling_avg
            })
            
            if episode_count % 5 == 0:
                print(f"Timestep: {timestep} | Episode: {episode_count} | Reward: {episode_reward:.2f}")
                
            obs, _ = env.reset()
            episode_reward = 0
            
    final_path =  "ppo_vizdoom_final.pth"
    torch.save(policy.state_dict(), final_path)
    print("Final Model Saved!")
    env.close()
    wandb.finish()
    print(f"VizDoom Training Completed")

In [17]:
train()

ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1334:(snd_func_refer) error evaluating name
ALSA lib conf.c:5178:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5701:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2664:(snd_pcm_open_noupdate) Unknown PCM default
wandb: WARNING `fps` argument does not affect the frame rate of the video when providing a file path or raw bytes.


Starting CartPole training for 100000 timesteps...
Using Device: cuda
-------Successfully logged videos for Episode 0----
Timestep: 108 | Episode: 5 | Reward: 20.00
Timestep: 210 | Episode: 10 | Reward: 16.00
Timestep: 290 | Episode: 15 | Reward: 30.00


wandb: WARNING `fps` argument does not affect the frame rate of the video when providing a file path or raw bytes.


Timestep: 379 | Episode: 20 | Reward: 10.00
-------Successfully logged videos for Episode 20----
Timestep: 478 | Episode: 25 | Reward: 30.00
Timestep: 588 | Episode: 30 | Reward: 12.00
Timestep: 725 | Episode: 35 | Reward: 35.00
Timestep: 830 | Episode: 40 | Reward: 12.00


wandb: WARNING `fps` argument does not affect the frame rate of the video when providing a file path or raw bytes.


...

-------Successfully logged videos for Episode 840----
Timestep: 89889 | Episode: 845 | Reward: 374.00
Saved checkpoint at step: 90000 -> ppo_cartpole_step_90000.pth
Timestep: 91237 | Episode: 850 | Reward: 182.00
Timestep: 93421 | Episode: 855 | Reward: 413.00
Timestep: 95817 | Episode: 860 | Reward: 500.00


wandb: WARNING `fps` argument does not affect the frame rate of the video when providing a file path or raw bytes.


-------Successfully logged videos for Episode 860----
Timestep: 97967 | Episode: 865 | Reward: 500.00
Timestep: 99874 | Episode: 870 | Reward: 500.00
Saved checkpoint at step: 100000 -> ppo_cartpole_step_100000.pth
Final Model Saved!


episode,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇██
episode_reward,▁▁▁▂▁▁▁▁▁▁▁▁▃▂▁▂▁▂▂▂▁▁▂▂▁▄▅▃▄▄▅▅█▅▅█▃██▃
reward_100_avg,▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▂▂▂▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▅▆▇▇▆▆█
timesteps,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
update/policy_loss,▂▃▄▃▄▅▂▆▇▂▆█▂▆▃▄▄▃▇▂▄▅▄▃▄▂▅▄▁▁▃▃▄▅▄▅▄▃▄▅
update_entropy,██▇▆▆▅▅▄▅▃▃▄▃▄▃▄▄▃▄▃▃▄▃▂▃▃▁▃▃▃▃▃▃▂▃▃▃▂▃▃
update_value_loss,▃▂▃▄▄▅▅▅▄▇▆▃█▃▅▃▃▄▂▃▆▂▄▄▁▁█▁▃▂▄▃▃▆▃▅▇█▆▇
episode,870
episode_reward,500
reward_100_avg,319.88
timesteps,98304


VizDoom Training Completed
